In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent / 'src'))

from data_preparation import (
    create_splits,
    get_split_statistics,
    save_all_splits,
    save_translation_subset,
    preprocess_for_transformers,
    preprocess_dataframe,
    AVAILABLE_LANGUAGES,
    TRAINING_LANGUAGES,
    ZERO_SHOT_LANGUAGE
)

from machine_translation import create_translated_dataset, load_translation_model, translate_texts_helsinki, select_translation_subset, translate_with_deepl

from paths import TRAIN_FILE, DEV_FILE

## 1. Load Raw Data and Initial Exploration


In [2]:
train_df = pd.read_csv(TRAIN_FILE)
dev_df = pd.read_csv(DEV_FILE)

print("Dataset Overview:")
print(train_df.columns.tolist())
print(train_df.dtypes.to_dict())

print("\nSample data:")
train_df.head(5)

Dataset Overview:
['id', 'text', 'label', 'origin', 'type', 'language', 'split']
{'id': <StringDtype(na_value=nan)>, 'text': <StringDtype(na_value=nan)>, 'label': dtype('float64'), 'origin': <StringDtype(na_value=nan)>, 'type': <StringDtype(na_value=nan)>, 'language': <StringDtype(na_value=nan)>, 'split': <StringDtype(na_value=nan)>}

Sample data:


,id,text,label,origin,type,language,split
0,ru_8872,знаете эти надписи на баночках с лекарствами з...,0.0,RuDReC,tweet,ru,SMM4H_26_train
1,ru_5231,"Знаю лично людей, которые лечились этим препар...",0.0,RuDReC,sentence,ru,SMM4H_26_train
2,zh_1298,我6月19号胃痛送去急诊科注射了两针颅通定针来止痛，结果6月23号就查出怀孕了，可是我月经应...,0.0,SMM4H_2026,120ask,zh,SMM4H_26_train
3,ja_13288,'ルルド先生が待機終了しました。06月21日22:30',0.0,SMM4H_2026,tweet,ja,SMM4H_26_train
4,en_14557,the following tweets are brought to you by ser...,0.0,previous_SMM4H,tweet,en,SMM4H_26_train


In [3]:
def analyze_dataset(df: pd.DataFrame, name: str = "Dataset") :
    stats = {
        'name': name,
        'total_samples': len(df),
        'ade_positive': int(df['label'].sum()),
        'ade_negative': int((df['label'] == 0).sum()),
        'ade_rate': df['label'].mean(),
        'languages': df['language'].value_counts().to_dict()
    }
    
    print(f"\n{name}:")
    print(f"   Total: {stats['total_samples']:,} samples")
    print(f"   ADE+: {stats['ade_positive']:,} ({stats['ade_rate']:.1%})")
    print(f"   ADE-: {stats['ade_negative']:,} ({1-stats['ade_rate']:.1%})")
    print(f"   Languages: {stats['languages']}")
    
analyze_dataset(train_df, "Full Training Set")
analyze_dataset(dev_df, "Full Dev Set")


Full Training Set:
   Total: 46,737 samples
   ADE+: 2,992 (6.4%)
   ADE-: 43,745 (93.6%)
   Languages: {'en': 17128, 'ja': 14208, 'ru': 10695, 'zh': 2248, 'de': 1482, 'fr': 976}

Full Dev Set:
   Total: 8,033 samples
   ADE+: 508 (6.3%)
   ADE-: 7,525 (93.7%)
   Languages: {'ja': 3045, 'ru': 2669, 'en': 888, 'de': 634, 'fr': 418, 'zh': 379}


In [4]:
train_counts = train_df['language'].value_counts()
dev_counts = dev_df['language'].value_counts()

comparison = pd.DataFrame({
    'Language': [AVAILABLE_LANGUAGES.get(l, l) for l in train_counts.index],
    'Train': train_counts.values,
    'Dev': [dev_counts.get(l, 0) for l in train_counts.index],
    'Train ADE Rate': [train_df[train_df['language']==l]['label'].mean() for l in train_counts.index],
    'Dev ADE Rate': [dev_df[dev_df['language']==l]['label'].mean() if l in dev_df['language'].values else 0 for l in train_counts.index]
})

print("\nLanguage Distribution Comparison:")
print(comparison.to_string(index=False))


Language Distribution Comparison:
Language  Train  Dev  Train ADE Rate  Dev ADE Rate
 English  17128  888        0.069944      0.068694
Japanese  14208 3045        0.023578      0.023645
 Russian  10695 2669        0.101169      0.101911
 Chinese   2248  379        0.099644      0.100264
  German   1482  634        0.056005      0.055205
  French    976  418        0.071721      0.071770


## 2. Language Selection 

### Select three languages
- **Two languages for training**: English (en) + Russian (ru)
- **One language for zero-shot evaluation**: German (de)

In [5]:
selected_langs = TRAINING_LANGUAGES + [ZERO_SHOT_LANGUAGE]
for lang in selected_langs:
    lang_train = train_df[train_df['language'] == lang]
    lang_dev = dev_df[dev_df['language'] == lang]
    role = "TRAINING" if lang in TRAINING_LANGUAGES else "ZERO-SHOT EVAL"
    print(f"\n{AVAILABLE_LANGUAGES[lang]} ({lang}) - {role}:")
    print(f"  Train samples: {len(lang_train):,} | Dev samples: {len(lang_dev):,}")
    print(f"  ADE rate: {lang_train['label'].mean():.1%}")


English (en) - TRAINING:
  Train samples: 17,128 | Dev samples: 888
  ADE rate: 7.0%

Russian (ru) - TRAINING:
  Train samples: 10,695 | Dev samples: 2,669
  ADE rate: 10.1%

German (de) - ZERO-SHOT EVAL:
  Train samples: 1,482 | Dev samples: 634
  ADE rate: 5.6%


## 3. Data Splits 

**Split Strategy:**
- Original **dev set** → redefined as **test set**
- New **validation set** → sampled from training data (10%)
- **Zero-shot test** → all German data (never seen during training)

In [6]:
splits = create_splits(
    train_df, 
    dev_df,
    training_langs=TRAINING_LANGUAGES,
    eval_lang=ZERO_SHOT_LANGUAGE,
    val_size=0.10, 
    random_state=42
)

In [7]:
stats_df = get_split_statistics(splits)
print("\n Split Statistics by Language:")
print(stats_df.to_string(index=False))

# Show dataset sizes
for lang in TRAINING_LANGUAGES:
    print(f"\n{AVAILABLE_LANGUAGES[lang]} ({lang}):")
    print(f"  Train: {len(splits[lang]['train'])} samples")
    print(f"  Val: {len(splits[lang]['val'])} samples")
    print(f"  Test: {len(splits[lang]['test'])} samples")

print(f"\nZero-shot ({AVAILABLE_LANGUAGES[ZERO_SHOT_LANGUAGE]}):")
print(f"  Test: {len(splits['zero_shot']['test'])} samples")


 Split Statistics by Language:
         Split Language  Samples  ADE+  ADE- ADE Rate
         Train  English    15415  1078 14337    6.99%
           Val  English     1713   120  1593    7.01%
          Test  English      888    61   827    6.87%
         Train  Russian     9625   974  8651   10.12%
           Val  Russian     1070   108   962   10.09%
          Test  Russian     2669   272  2397   10.19%
Zero-Shot Test   German     2116   118  1998    5.58%

English (en):
  Train: 15415 samples
  Val: 1713 samples
  Test: 888 samples

Russian (ru):
  Train: 9625 samples
  Val: 1070 samples
  Test: 2669 samples

Zero-shot (German):
  Test: 2116 samples


## 4. Text Preprocessing for Transformer Models


In [8]:
# Apply preprocessing to all splits
for lang in TRAINING_LANGUAGES:
    splits[lang]['train'] = preprocess_dataframe(splits[lang]['train'])
    splits[lang]['val'] = preprocess_dataframe(splits[lang]['val'])
    splits[lang]['test'] = preprocess_dataframe(splits[lang]['test'])

splits['zero_shot']['test'] = preprocess_dataframe(splits['zero_shot']['test'])


for lang in TRAINING_LANGUAGES + [ZERO_SHOT_LANGUAGE]:
    if lang in TRAINING_LANGUAGES:
        sample = splits[lang]['train'].head(1).iloc[0]
    else:
        sample = splits['zero_shot']['test'].head(1).iloc[0]
    print(f"[{lang.upper()}] Original: {sample['text'][:100]}...")
    print(f"[{lang.upper()}] Cleaned:  {sample['text_clean'][:100]}...")
    print()

[EN] Original: Hyper af! I feel like a bird on vyvanse...
[EN] Cleaned:  Hyper af! I feel like a bird on vyvanse...

[RU] Original: Смотрю попеременно на лоперамид и суматриптан, представила себя, сидящей на толчке с мигренью и реши...
[RU] Cleaned:  Смотрю попеременно на лоперамид и суматриптан, представила себя, сидящей на толчке с мигренью и реши...

[DE] Original: Hallo <user>, der lange Aufenthalt hier im Forum bildet ;-) und ich habe mich ja auch aus Eigeninter...
[DE] Cleaned:  Hallo <[USER]>, der lange Aufenthalt hier im Forum bildet ;-) und ich habe mich ja auch aus Eigenint...



## 5. Machine Translation: English → German (This part moved to translation.ipynb)

Translate a subset of English training data to German for data augmentation.

In [19]:
en_subset = select_translation_subset(
    splits['en']['train'],
    n_samples=2000,
    random_state=42
)

print(f"\nTranslation Subset Statistics:")
print(f"   Total samples: {len(en_subset)}")
print(f"   ADE+: {int(en_subset['label'].sum())} ({en_subset['label'].mean():.1%})")
print(f"   ADE-: {int((en_subset['label']==0).sum())} ({(en_subset['label']==0).mean():.1%})")

for i, row in en_subset.head(3).iterrows():
    print(f"\n  [{int(row['label'])}] {row['text_clean'][:100]}...")


Selected 2000 English samples for translation:
   ADE+: 140 (7.0%)
   ADE-: 1860 (93.0%)

Translation Subset Statistics:
   Total samples: 2000
   ADE+: 140 (7.0%)
   ADE-: 1860 (93.0%)

  [0] bury me in a bathtub filled with Paxil...

  [0] [USER] happy Mothers Day to you also! My son has been doing very well. It's Enbrel night for him als...

  [0] lncRNA study from The Medical School of Jiangnan University: Downregulation of lncRNA GAS5 Confers T...


In [10]:
# Load model (this will download on first run)
tokenizer, model = load_translation_model("Helsinki-NLP/opus-mt-en-de")

Loading translation model: Helsinki-NLP/opus-mt-en-de


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded successfully!


In [12]:
texts_to_translate = en_subset['text_clean'].tolist()

# Translate using Helsinki model
translations = translate_texts_helsinki(
    texts_to_translate,
    tokenizer=tokenizer,
    model=model,
    batch_size=16
)

print(f"\nTranslated {len(translations)} texts")

# Show sample translations
print("\nSample translations:")
print("-"*70)
for i in range(min(5, len(translations))):
    print(f"\nEN: {texts_to_translate[i][:100]}...")
    print(f"DE: {translations[i][:100]}...")

# Create translated dataset
translated_df = create_translated_dataset(en_subset, translations, target_lang='de')
print(f"\nTranslated dataset shape: {translated_df.shape}")
print(f"   Columns: {translated_df.columns.tolist()}")

Translating 2000 texts (device: cpu)...


Translating: 100%|██████████| 125/125 [11:55<00:00,  5.73s/it]

Translation complete!

Translated 2000 texts

Sample translations:
----------------------------------------------------------------------

EN: bury me in a bathtub filled with Paxil...
DE: begraben mich in einer Badewanne gefüllt mit Paxil...

EN: [USER] happy Mothers Day to you also! My son has been doing very well. It's Enbrel night for him als...
DE: Auch dir ist ein glücklicher Muttertag! Mein Sohn hat es sehr gut gemacht. Es ist auch Enbrel-Nacht ...

EN: lncRNA study from The Medical School of Jiangnan University: Downregulation of lncRNA GAS5 Confers T...
DE: lncRNA-Studie von der Medizinischen Fakultät der Jiangnan Universität: Downregulation von lncRNA GAS...

EN: depression hurts :( cymbalta can help (:...
DE: Depressionen schmerzen :( Cymbalta kann helfen (:...

EN: Zanamivir and oseltamivir are active against both influenza A and B...
DE: Zanamivir und Oseltamivir wirken sowohl gegen Influenza A als auch gegen B....

Translated dataset shape: (2000, 13)
   Columns: ['id', '

## 6. Save All Processed Data

In [18]:
save_all_splits(splits)


✅ All splits saved to /Users/smelihportakal/PythonProject/AdvancedNLP-2/task1/data/processed


In [ ]:
save_translation_subset(en_subset, translated_df)


Translation files saved to /Users/smelihportakal/PythonProject/AdvancedNLP-2/task1/data/processed/translation
